In [46]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
customer_id = df['customerID']
data = df.drop(['customerID'], axis=1)

cols = ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

# data['gender'] = data['gender'].map({'Male': 1, 'Female': 0})
# cols_without_yesno = []

# print(df['Partner'].value_counts().index.tolist())    # check output, can be misarranged

# for col in cols:
    
#     if df[col].dtype == 'object':
#         # print(df[col].value_counts())
#         pass

#     if df[col].value_counts().index.tolist() != ['No', 'Yes']:
#         cols_without_yesno.append(col)

#     if df[col].value_counts().index.tolist() == ['No', 'Yes']:
#         df[col] = df[col].map({'Yes': 1, 'No': 0})

data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

data.sample(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
2220,Male,0,Yes,No,72,Yes,No,DSL,Yes,Yes,No,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),79.35,5753.25,No
6013,Female,1,Yes,No,50,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Electronic check,100.65,5189.75,No
5067,Female,0,Yes,Yes,30,Yes,No,DSL,Yes,No,No,Yes,No,Yes,Month-to-month,No,Bank transfer (automatic),66.30,1923.50,No
3767,Male,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),110.90,8240.85,No
5553,Male,0,No,No,1,Yes,No,DSL,No,No,No,No,No,Yes,Month-to-month,Yes,Mailed check,55.70,55.70,Yes


In [47]:
cols_without_yesno = []
cols_with_yesno = ['gender',]

for col in cols:
    # print(data[col].nunique())
    if data[col].nunique() > 2:
        cols_without_yesno.append(col)
    else :
        cols_with_yesno.append(col)
cols_without_yesno

['MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaymentMethod']

In [48]:
from sklearn.model_selection import train_test_split

x = data.drop(['Churn'], axis=1)
y = data['Churn']
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, test_size=0.2)

In [49]:
x_train.sample(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
6013,Female,1,Yes,No,50,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Electronic check,100.65,5189.75
2447,Male,0,No,No,7,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,74.90,490.55
3949,Female,0,No,Yes,18,Yes,Yes,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,57.65,992.70
2935,Male,0,Yes,No,71,Yes,No,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),86.10,6045.90
3630,Female,0,No,No,4,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),20.95,85.50


### Transformers

In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# trf1 = ColumnTransformer(transformers=[
#     ('impute_totalcharges', SimpleImputer(), [18])
# ], remainder='passthrough')

# trf2 = ColumnTransformer(transformers=[
#     ("mapping", OrdinalEncoder(), [0])
# ], remainder='passthrough')

# trf3 = ColumnTransformer(transformers=[
#     ("ohe", OneHotEncoder(handle_unknown="ignore"), [6, 7, 8, 9, 10, 11, 12, 13, 14, 16])
# ], remainder='passthrough')

# trf4 = SelectKBest(k=8)

# trf5 = RandomForestClassifier()


processor = ColumnTransformer(transformers=[
    ("Gender_encoding", OrdinalEncoder(), cols_with_yesno),
    ("Missing_values", SimpleImputer(), ['TotalCharges']),
    ("OHE", OneHotEncoder(), cols_without_yesno)
], remainder='passthrough')

In [51]:
from sklearn.pipeline import make_pipeline, Pipeline

pipe = Pipeline([
    # ('trf1',trf1),
    # ('trf2',trf2),
    # ('trf3',trf3),
    # ('trf4',trf4),
    # ('trf5',trf5)
    ('processor', processor),
    ('select_features', SelectKBest(k=8)),
    ('modeling', RandomForestClassifier())
])

# pipe = make_pipeline(trf1, trf2, trf3, trf4, trf5)
pipe.fit(x_train, y_train)

,steps,"[('processor', ...), ('select_features', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Gender_encoding', ...), ('Missing_values', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [52]:
from sklearn.metrics import accuracy_score, roc_auc_score
y_pred = pipe.predict(x_test)
print(f"accuracy {accuracy_score(y_test, y_pred)}")
print(f"AUC ROC {roc_auc_score(y_test, y_pred)}")

accuracy 0.7572746628814763


ValueError: could not convert string to float: 'Yes'